In [4]:
# make sure jupyter server is installed in the environment
# then install dependencies
%pip install pandas nltk scikit-learn numpy matplotlib --quiet

from config import get_merged_dataframe
from main import configure

configure()

df = get_merged_dataframe(
    './data/processedNegative.csv',
    './data/processedPositive.csv',
    './data/processedNeutral.csv',
)

df.sample(10).reset_index(drop=True)

Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt_tab to /home/samy/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/samy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/samy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,tweet,sentiment
0,Administrators named for cricket body.,neutral
1,Thanks smile much better today and thanks for...,positive
2,Modi says people don't want handouts,neutral
3,before I think of them :)KISSES TheFashionIcon,positive
4,I'm one year younger to one year elder to in,neutral
5,Thanks for the recent follow Happy to connect ...,positive
6,Pretty much only the Twitters and it doesn't r...,negative
7,beautiful set,positive
8,Pls RT[NCT FIC] Love Song,positive
9,space for debate and dissent continues to shr...,neutral


#### Different Machine Learning methods

In [ ]:
from tokenizer import lemmatize_tokens, stem_tokens
from cleaning import clean_tweets
from vectorizer import tfidf_vectorize, count_vectorize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

config = {
    "cleaning": {
        "Default Cleaning": clean_tweets,
    },
    "tokenization": {
        "Lemmitization": lemmatize_tokens,
        "Stemming": stem_tokens,
    },
    "vectorization": {
        "TF-IDF": tfidf_vectorize,
        "Count": count_vectorize,
    },
    "classifiers": [
        LogisticRegression(),
        RandomForestClassifier(n_estimators=100, random_state=42),
    ],
}

In [12]:
from train import train_model, evaluate_model
import warnings
warnings.filterwarnings('ignore', message='The parameter.*token_pattern.*will not be used')

# Grid Search over all combinations
for model in config['classifiers']:
    # For each vectorization technique
    for vectorizer_name, vectorizer_func in config['vectorization'].items():
        # For each tokenization technique
        for tokenizer_name, tokenizer_func in config['tokenization'].items():
            # For each cleaning technique
            for cleaner_name, cleaner_func in config['cleaning'].items():
                # Print current combination
                print(f"Training {model.__class__.__name__}, {vectorizer_name}, {tokenizer_name} {cleaner_name}... ")
                model, _, x_test, y_test = train_model(
                    df,
                    tweet_column='tweet',
                    sentiment_column='sentiment',
                    cleaner=cleaner_func,
                    tokenizer=tokenizer_func,
                    vectorizer=vectorizer_func,
                    classifier=model,
                )
                acc = evaluate_model(model, x_test, y_test)
                # multiply by 100 to get percentage
                print(f"Accuracy: {acc:.2%}")
                print("-" * 50)

Training LogisticRegression, TF-IDF Vectorizer, Lemmitization Default Cleaning... 
Accuracy: 89.41%
--------------------------------------------------
Training LogisticRegression, TF-IDF Vectorizer, Stemming Default Cleaning... 
Accuracy: 90.18%
--------------------------------------------------
Training LogisticRegression, Count Vectorizer, Lemmitization Default Cleaning... 
Accuracy: 90.83%
--------------------------------------------------
Training LogisticRegression, Count Vectorizer, Stemming Default Cleaning... 
Accuracy: 91.09%
--------------------------------------------------
Training RandomForestClassifier, TF-IDF Vectorizer, Lemmitization Default Cleaning... 
Accuracy: 91.09%
--------------------------------------------------
Training RandomForestClassifier, TF-IDF Vectorizer, Stemming Default Cleaning... 
Accuracy: 91.34%
--------------------------------------------------
Training RandomForestClassifier, Count Vectorizer, Lemmitization Default Cleaning... 
Accuracy: 91.47%


### Cosine Similarity